In [1]:
#1 — Imports
import os
import csv
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb
from skimage.metrics import peak_signal_noise_ratio as psnr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sys
sys.path.append(os.path.expanduser("~/colorizer_gan/notebooks"))

from dataset import ColorizationDataset
from model   import Generator, Discriminator, init_weights

In [2]:
#2 — Config
# Paths
DATASET_PATH  = "/mnt/hdd/colorizer_gan/datasets/coco"
TRAIN_PATH    = os.path.join(DATASET_PATH, "color/train2017")
VAL_PATH      = os.path.join(DATASET_PATH, "val/val2017")
OUTPUT_PATH   = os.path.expanduser("~/colorizer_gan/outputs")
SAMPLE_PATH   = os.path.join(OUTPUT_PATH, "samples")
PLOT_PATH     = os.path.join(OUTPUT_PATH, "plots")
LOG_PATH      = os.path.expanduser("~/colorizer_gan/logs")
MODEL_PATH    = os.path.expanduser("~/colorizer_gan/models")

# Training config
IMAGE_SIZE    = 256
BATCH_SIZE    = 64
NUM_WORKERS   = 16
PIN_MEMORY    = True
EPOCHS        = 100
LR            = 0.0002
BETA1         = 0.5
BETA2         = 0.999
LAMBDA_L1     = 100.0
SAVE_EVERY    = 5        # save sample images every N epochs
CHECKPOINT_EVERY = 10   # save model checkpoint every N epochs
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create directories
for path in [OUTPUT_PATH, SAMPLE_PATH, PLOT_PATH, LOG_PATH, MODEL_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"Device:       {DEVICE}")
print(f"Epochs:       {EPOCHS}")
print(f"Batch size:   {BATCH_SIZE}")
print(f"LR:           {LR}")
print(f"Lambda L1:    {LAMBDA_L1}")
print(f"Save every:   {SAVE_EVERY} epochs")

Device:       cuda
Epochs:       100
Batch size:   64
LR:           0.0002
Lambda L1:    100.0
Save every:   5 epochs


In [3]:
#3 load dataset
train_dataset = ColorizationDataset(TRAIN_PATH, IMAGE_SIZE, split="train")
val_dataset   = ColorizationDataset(VAL_PATH,   IMAGE_SIZE, split="val")

train_loader  = DataLoader(
    train_dataset,
    batch_size   = BATCH_SIZE,
    shuffle      = True,
    num_workers  = NUM_WORKERS,
    pin_memory   = PIN_MEMORY,
    drop_last    = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size   = BATCH_SIZE,
    shuffle      = False,
    num_workers  = NUM_WORKERS,
    pin_memory   = PIN_MEMORY,
    drop_last    = False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

 train dataset loaded: 118287 images
 val dataset loaded: 5000 images
Train batches: 1848
Val batches:   79


In [4]:
#4 initialize models and optimizers
# Initialize models
G = init_weights(Generator()).to(DEVICE)
D = init_weights(Discriminator()).to(DEVICE)

# Optimizers
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2))

# Learning rate schedulers — linear decay over all epochs
scheduler_G = torch.optim.lr_scheduler.LinearLR(
    opt_G, start_factor=1.0, end_factor=0.0, total_iters=EPOCHS
)
scheduler_D = torch.optim.lr_scheduler.LinearLR(
    opt_D, start_factor=1.0, end_factor=0.0, total_iters=EPOCHS
)

print(f" Generator initialized")
print(f" Discriminator initialized")
print(f" Optimizers ready")
print(f" Schedulers ready")
print(f"\nGenerator parameters:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in D.parameters()):,}")

 Generator initialized
 Discriminator initialized
 Optimizers ready
 Schedulers ready

Generator parameters:     54,410,434
Discriminator parameters: 2,765,633


In [5]:
#5 — Loss Functions
adversarial_loss = nn.MSELoss()    # LSGAN — more stable than BCE
l1_loss          = nn.L1Loss()     # content preservation

# Fixed labels
real_label = 1.0
fake_label = 0.0

print(" Loss functions ready")
print(f"   Adversarial: MSE (LSGAN)")
print(f"   Content:     L1  (weight={LAMBDA_L1})")

 Loss functions ready
   Adversarial: MSE (LSGAN)
   Content:     L1  (weight=100.0)


In [6]:
#6  — Training Log Setup (CSV + live table)
LOG_FILE = os.path.join(LOG_PATH, "training_log.csv")

# Create CSV with headers
with open(LOG_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch", "loss_G", "loss_D", "loss_G_adv",
        "loss_G_l1", "val_loss_G", "val_psnr",
        "epoch_time_s", "lr_G", "lr_D"
    ])

print(f" Training log created at {LOG_FILE}")
print(f"\n{'='*85}")
print(f"  {'Epoch':>6} | {'Loss G':>8} | {'Loss D':>8} | {'G Adv':>8} | "
      f"{'G L1':>8} | {'Val G':>8} | {'PSNR':>7} | {'Time':>7}")
print(f"{'='*85}")

 Training log created at /home/61447093s/colorizer_gan/logs/training_log.csv

   Epoch |   Loss G |   Loss D |    G Adv |     G L1 |    Val G |    PSNR |    Time


In [7]:
#7  — Save Sample Images Function
def save_samples(G, val_loader, epoch, device, n=4):
    G.eval()
    L_batch, AB_batch = next(iter(val_loader))
    L_batch  = L_batch[:n].to(device)
    AB_batch = AB_batch[:n]

    with torch.no_grad():
        AB_pred = G(L_batch).cpu()

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 4))
    fig.suptitle(f"Epoch {epoch:03d} — Grayscale | Predicted | Ground Truth",
                 fontsize=13, fontweight="bold")

    for i in range(n):
        # Grayscale input
        L_np = (L_batch[i].cpu().squeeze().numpy() + 1.0) * 50.0

        # Predicted color
        AB_pred_np = AB_pred[i].permute(1, 2, 0).numpy() * 128.0
        lab_pred   = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32")
        lab_pred[:, :, 0]  = L_np
        lab_pred[:, :, 1:] = AB_pred_np
        rgb_pred   = np.clip(lab2rgb(lab_pred), 0, 1)

        # Ground truth color
        AB_real_np = AB_batch[i].permute(1, 2, 0).numpy() * 128.0
        lab_real   = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32")
        lab_real[:, :, 0]  = L_np
        lab_real[:, :, 1:] = AB_real_np
        rgb_real   = np.clip(lab2rgb(lab_real), 0, 1)

        axes[i, 0].imshow(L_np,    cmap="gray")
        axes[i, 1].imshow(rgb_pred)
        axes[i, 2].imshow(rgb_real)

        axes[i, 0].set_title("Grayscale Input")
        axes[i, 1].set_title("Predicted Color")
        axes[i, 2].set_title("Ground Truth")

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    save_path = os.path.join(SAMPLE_PATH, f"epoch_{epoch:03d}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    G.train()

In [8]:
#8  — Plot Loss Curves Function
def plot_loss_curves(history):
    epochs = history["epoch"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training Loss Curves", fontsize=14, fontweight="bold")

    # Generator losses
    axes[0].plot(epochs, history["loss_G"],     label="Total G Loss",   color="#4C72B0", linewidth=2)
    axes[0].plot(epochs, history["loss_G_adv"], label="G Adversarial",  color="#DD8452", linewidth=1.5, linestyle="--")
    axes[0].plot(epochs, history["loss_G_l1"],  label="G L1",           color="#55A868", linewidth=1.5, linestyle="--")
    axes[0].plot(epochs, history["val_loss_G"], label="Val G Loss",     color="#C44E52", linewidth=2,   linestyle=":")
    axes[0].set_title("Generator Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Discriminator loss
    axes[1].plot(epochs, history["loss_D"], label="D Loss", color="#8172B2", linewidth=2)
    axes[1].set_title("Discriminator Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_PATH, "loss_curves.png"), dpi=150, bbox_inches="tight")
    plt.close()

In [9]:
#9  — Plot PSNR Curve Function
def plot_psnr_curve(history):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(history["epoch"], history["val_psnr"],
            color="#4C72B0", linewidth=2, label="Validation PSNR")
    ax.set_title("Validation PSNR per Epoch", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("PSNR (dB)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Annotate best PSNR
    best_epoch = history["epoch"][history["val_psnr"].index(max(history["val_psnr"]))]
    best_psnr  = max(history["val_psnr"])
    ax.annotate(f"Best: {best_psnr:.2f} dB\n(Epoch {best_epoch})",
                xy=(best_epoch, best_psnr),
                xytext=(best_epoch + 2, best_psnr - 1),
                arrowprops=dict(arrowstyle="->", color="red"),
                color="red", fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_PATH, "psnr_curve.png"), dpi=150, bbox_inches="tight")
    plt.close()

In [10]:
#10 — Training Step Function (single batch)
def train_step(L, AB_real, G, D, opt_G, opt_D):
    L       = L.to(DEVICE)
    AB_real = AB_real.to(DEVICE)

    # ---- Train Discriminator ----
    opt_D.zero_grad()

    AB_fake    = G(L).detach()          # detach so G doesn't get gradients here
    D_real     = D(L, AB_real)
    D_fake     = D(L, AB_fake)

    real_labels = torch.ones_like(D_real)  * real_label
    fake_labels = torch.zeros_like(D_fake) * fake_label

    loss_D_real = adversarial_loss(D_real, real_labels)
    loss_D_fake = adversarial_loss(D_fake, fake_labels)
    loss_D      = (loss_D_real + loss_D_fake) * 0.5

    loss_D.backward()
    opt_D.step()

    # ---- Train Generator ----
    opt_G.zero_grad()

    AB_fake  = G(L)
    D_fake   = D(L, AB_fake)

    loss_G_adv = adversarial_loss(D_fake, torch.ones_like(D_fake))
    loss_G_l1  = l1_loss(AB_fake, AB_real) * LAMBDA_L1
    loss_G     = loss_G_adv + loss_G_l1

    loss_G.backward()
    opt_G.step()

    return (
        loss_G.item(),
        loss_D.item(),
        loss_G_adv.item(),
        loss_G_l1.item()
    )

In [11]:
#11 — Validation Step Function
def validate(G, val_loader, device, num_batches=20):
    G.eval()
    total_loss_G = 0.0
    total_psnr   = 0.0
    count        = 0

    with torch.no_grad():
        for i, (L, AB_real) in enumerate(val_loader):
            if i >= num_batches:
                break

            L       = L.to(device)
            AB_real = AB_real.to(device)
            AB_fake = G(L)

            # Validation L1 loss
            loss_G = l1_loss(AB_fake, AB_real) * LAMBDA_L1
            total_loss_G += loss_G.item()

            # PSNR — convert back to RGB for proper measurement
            for j in range(L.shape[0]):
                L_np       = (L[j].cpu().squeeze().numpy() + 1.0) * 50.0
                AB_pred_np = AB_fake[j].cpu().permute(1, 2, 0).numpy() * 128.0
                AB_real_np = AB_real[j].cpu().permute(1, 2, 0).numpy() * 128.0

                lab_pred             = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32")
                lab_pred[:, :, 0]    = L_np
                lab_pred[:, :, 1:]   = AB_pred_np

                lab_real             = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32")
                lab_real[:, :, 0]    = L_np
                lab_real[:, :, 1:]   = AB_real_np

                rgb_pred = np.clip(lab2rgb(lab_pred), 0, 1)
                rgb_real = np.clip(lab2rgb(lab_real), 0, 1)

                total_psnr += psnr(rgb_real, rgb_pred, data_range=1.0)
                count      += 1

    G.train()
    avg_loss = total_loss_G / num_batches
    avg_psnr = total_psnr  / count if count > 0 else 0.0

    return avg_loss, avg_psnr

In [12]:
#sanity cgheck
# Test one full training step
L, AB_real = next(iter(train_loader))
loss_G, loss_D, loss_G_adv, loss_G_l1 = train_step(L, AB_real, G, D, opt_G, opt_D)
print(f"loss_G:     {loss_G:.4f}")
print(f"loss_D:     {loss_D:.4f}")
print(f"loss_G_adv: {loss_G_adv:.4f}")
print(f"loss_G_l1:  {loss_G_l1:.4f}")

# Test validation
val_loss, val_psnr = validate(G, val_loader, DEVICE)
print(f"val_loss:   {val_loss:.4f}")
print(f"val_psnr:   {val_psnr:.2f} dB")

loss_G:     49.6109
loss_D:     1.5070
loss_G_adv: 28.9436
loss_G_l1:  20.6673


val_loss:   7.7375
val_psnr:   23.18 dB


In [ ]:
#12 — Training Loop (full)
# History tracking for plots
history = {
    "epoch":      [],
    "loss_G":     [],
    "loss_D":     [],
    "loss_G_adv": [],
    "loss_G_l1":  [],
    "val_loss_G": [],
    "val_psnr":   [],
}

best_psnr   = 0.0
start_total = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start  = time.time()
    G.train()
    D.train()

    # Accumulate batch losses
    epoch_loss_G     = 0.0
    epoch_loss_D     = 0.0
    epoch_loss_G_adv = 0.0
    epoch_loss_G_l1  = 0.0

    for batch_idx, (L, AB_real) in enumerate(train_loader):
        loss_G, loss_D, loss_G_adv, loss_G_l1 = train_step(
            L, AB_real, G, D, opt_G, opt_D
        )
        epoch_loss_G     += loss_G
        epoch_loss_D     += loss_D
        epoch_loss_G_adv += loss_G_adv
        epoch_loss_G_l1  += loss_G_l1

    # Average losses over batches
    n_batches        = len(train_loader)
    epoch_loss_G     /= n_batches
    epoch_loss_D     /= n_batches
    epoch_loss_G_adv /= n_batches
    epoch_loss_G_l1  /= n_batches

    # Validation
    val_loss_G, val_psnr = validate(G, val_loader, DEVICE)

    # Step schedulers
    scheduler_G.step()
    scheduler_D.step()

    # Epoch time
    epoch_time = time.time() - epoch_start

    # Save to history
    history["epoch"].append(epoch)
    history["loss_G"].append(epoch_loss_G)
    history["loss_D"].append(epoch_loss_D)
    history["loss_G_adv"].append(epoch_loss_G_adv)
    history["loss_G_l1"].append(epoch_loss_G_l1)
    history["val_loss_G"].append(val_loss_G)
    history["val_psnr"].append(val_psnr)

    # Log to CSV
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            f"{epoch_loss_G:.4f}",
            f"{epoch_loss_D:.4f}",
            f"{epoch_loss_G_adv:.4f}",
            f"{epoch_loss_G_l1:.4f}",
            f"{val_loss_G:.4f}",
            f"{val_psnr:.2f}",
            f"{epoch_time:.1f}",
            f"{opt_G.param_groups[0]['lr']:.6f}",
            f"{opt_D.param_groups[0]['lr']:.6f}",
        ])

    # Print epoch summary
    print(f"  {epoch:>6} | {epoch_loss_G:>8.4f} | {epoch_loss_D:>8.4f} | "
          f"{epoch_loss_G_adv:>8.4f} | {epoch_loss_G_l1:>8.4f} | "
          f"{val_loss_G:>8.4f} | {val_psnr:>7.2f} | {epoch_time:>6.1f}s")

    # Save sample images
    if epoch % SAVE_EVERY == 0 or epoch == 1:
        save_samples(G, val_loader, epoch, DEVICE)
        plot_loss_curves(history)
        plot_psnr_curve(history)

    # Save best model
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save({
            "epoch":       epoch,
            "G_state":     G.state_dict(),
            "D_state":     D.state_dict(),
            "opt_G_state": opt_G.state_dict(),
            "opt_D_state": opt_D.state_dict(),
            "best_psnr":   best_psnr,
        }, os.path.join(MODEL_PATH, "best_model.pth"))

    # Save checkpoint
    if epoch % CHECKPOINT_EVERY == 0:
        torch.save({
            "epoch":       epoch,
            "G_state":     G.state_dict(),
            "D_state":     D.state_dict(),
            "opt_G_state": opt_G.state_dict(),
            "opt_D_state": opt_D.state_dict(),
            "history":     history,
        }, os.path.join(MODEL_PATH, f"checkpoint_epoch_{epoch:03d}.pth"))
        print(f"  Checkpoint saved at epoch {epoch}")

total_time = time.time() - start_total
print(f"\n{'='*85}")
print(f"  Training complete! Total time: {total_time/3600:.2f} hours")
print(f"  Best PSNR: {best_psnr:.2f} dB")
print(f"{'='*85}")

In [ ]:
#13 — Plot Final Results
def plot_final_results(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Final Training Results", fontsize=14, fontweight="bold")

    epochs = history["epoch"]

    # Total losses
    axes[0].plot(epochs, history["loss_G"], label="Train G", color="#4C72B0", linewidth=2)
    axes[0].plot(epochs, history["val_loss_G"], label="Val G", color="#C44E52",
                linewidth=2, linestyle="--")
    axes[0].set_title("Generator Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Discriminator loss
    axes[1].plot(epochs, history["loss_D"], color="#8172B2", linewidth=2)
    axes[1].set_title("Discriminator Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.3)

    # PSNR
    axes[2].plot(epochs, history["val_psnr"], color="#55A868", linewidth=2)
    axes[2].set_title("Validation PSNR")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("PSNR (dB)")
    axes[2].grid(True, alpha=0.3)

    # Annotate best PSNR
    best_epoch = history["epoch"][history["val_psnr"].index(max(history["val_psnr"]))]
    best_psnr  = max(history["val_psnr"])
    axes[2].annotate(
        f"Best: {best_psnr:.2f} dB\n(Epoch {best_epoch})",
        xy     = (best_epoch, best_psnr),
        xytext = (best_epoch + 2, best_psnr - 1),
        arrowprops = dict(arrowstyle="->", color="red"),
        color="red", fontsize=10
    )

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_PATH, "final_results.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print(" Final results saved to outputs/plots/final_results.png")

    # Print final summary table
    print(f"\n{'='*55}")
    print(f"  {'FINAL TRAINING SUMMARY':^51}")
    print(f"{'='*55}")
    print(f"  {'Total epochs':<25} {EPOCHS}")
    print(f"  {'Best PSNR':<25} {max(history['val_psnr']):.2f} dB")
    print(f"  {'Best epoch':<25} {history['epoch'][history['val_psnr'].index(max(history['val_psnr']))]}")
    print(f"  {'Final G loss':<25} {history['loss_G'][-1]:.4f}")
    print(f"  {'Final D loss':<25} {history['loss_D'][-1]:.4f}")
    print(f"  {'Final val PSNR':<25} {history['val_psnr'][-1]:.2f} dB")
    print(f"{'='*55}")

plot_final_results(history)